# 40. Supplement tables

Writes the supplement's LaTeX tables (`tables/S*.tex`, read by `NewGlenn_explosion_supplement.tex`) from the
CSV products of notebooks 21–38 in `cfg.OUT_DIR`. Rerun after any of those notebooks changes.

In [1]:
import numpy as np
import pandas as pd
import newglenn_config as cfg

O = cfg.OUT_DIR
TAB = cfg.PAPER_DIR / 'tables'
TAB.mkdir(exist_ok=True)

def write(name, df, caption, label, colfmt, note=None, size='small'):
    body = df.to_latex(index=False, escape=False, column_format=colfmt, na_rep='--')
    tex = (f'\\begin{{table}}[htbp]\n  \\centering\n  \\caption{{{caption}}}\n  \\label{{{label}}}\n'
           f'  \\{size}\n  \\fitwidth{{%\n{body}  }}\n')
    if note:
        tex += f'  \\par\\smallskip\\parbox{{0.95\\textwidth}}{{\\footnotesize {note}}}\n'
    tex += '\\end{table}\n'
    (TAB / f'{name}.tex').write_text(tex)
    print(name, df.shape)

def f(x, d=1):
    return '--' if pd.isna(x) else f'{x:.{d}f}'

## S1 Stations and S2 channel status

In [2]:
st = pd.read_csv(O / '21_station_map' / 'station_table.csv')
cs = pd.read_csv(O / '22_data_overview' / 'channel_status.csv')
fs = cs.groupby('station').fs.first()
s1 = pd.DataFrame({'Station': st.station, 'Lat ($^\\circ$N)': st.lat.map(lambda x: f(x, 4)),
                   'Lon ($^\\circ$E)': st.lon.map(lambda x: f(x, 4)), 'Distance (km)': st.distance_km.map(lambda x: f(x, 2)),
                   'Azimuth ($^\\circ$)': st.azimuth_deg.map(lambda x: f(x, 1)),
                   'Samples s$^{-1}$': st.station.map(fs).map(lambda x: f(x, 0)), 'Status': st.status})
write('S1_stations', s1, 'Stations recording on 29~May~2026: coordinates, distance and azimuth from LC-36 '
      '(28.4706$^\\circ$N, 80.5422$^\\circ$W), sampling rate and status for the explosion [21, 22].',
      'tab:s_stations', 'lrrrrrp{5.2cm}')

CODE = {'ok': 'ok', 'no sensor': 'none', 'railed/clipped': 'rail', 'faulty Centaur': 'fault',
        'no sensor (and faulty Centaur)': 'none'}
noise = cs.noise_counts.map(lambda x: f'{x:,.0f}' if x >= 100 else f'{x:.1f}')
code_ = cs.status.map(CODE).fillna(cs.status)
code_[(cs.station == 'B20') & cs.channel.isin(['DH1', 'DH2'])] = 'same'
grid = cs.assign(code=code_ + ' (' + noise + ')')
grid = grid.pivot(index='station', columns='channel', values='code').reindex(columns=['DHZ', 'DH1', 'DH2', 'DD0', 'DD1', 'DD2', 'DDF'])
grid = grid.loc[st.station].reset_index().rename(columns={'station': 'Station'})
write('S2_channel_status', grid,
      'Channel status for the explosion, with the pre-event noise in raw counts (rms, 250--20~s before the origin) in '
      'brackets [22]. ok: working; none: no sensor connected (digitizer noise only); rail: within 20\\% of the 24-bit '
      'rail during the event (Trillium mass at its stop, or a clipped sensor); same: records the same signal as the other '
      'horizontal (B20 DH1 and DH2 correlate at 0.9999 all year). B03\\textquoteright s Centaur channels '
      'are also known to be faulty.',
      'tab:s_channels', 'llllllll', size='scriptsize')

S1_stations (7, 7)
S2_channel_status (7, 8)


## S3 Weather and airwave speeds, S4 onsets and origin fits

In [3]:
w = pd.read_csv(O / '24_origin_time' / 'weather_at_origin.csv')
s3 = pd.DataFrame({'Station': w.station, 'Obs.': w.n.astype(int), 'T ($^\\circ$C)': w.temperature_c.map(f),
                   'RH (\\%)': w.rh.map(f), 'Pressure (hPa)': (w.pressure_pa / 100).map(f),
                   'Wind (m~s$^{-1}$)': w.wind_speed_mps.map(f), 'Wind from ($^\\circ$)': w.wind_from_deg.map(lambda x: f(x, 0))})
sp = pd.read_csv(O / '24_origin_time' / 'airwave_speeds.csv').set_index('station')
use = ['B20', 'B14', 'B29', 'B05']
s3b = pd.DataFrame({'Station': use, 'Azimuth ($^\\circ$)': sp.loc[use, 'azimuth_deg'].map(f).values,
                    '$c_0$ (m~s$^{-1}$)': sp.loc[use, 'c0_mps'].map(f).values,
                    'Wind along ray (m~s$^{-1}$)': sp.loc[use, 'wind_along_ray_mps'].map(lambda x: f(x, 2)).values,
                    '$c_\\mathrm{eff}$ (m~s$^{-1}$)': sp.loc[use, 'c_eff_mps'].map(f).values,
                    'Travel time (s)': sp.loc[use, 'travel_time_s'].map(lambda x: f(x, 2)).values})
write('S3_weather', s3, 'Surface weather within 30~minutes of the origin (Iowa Environmental Mesonet ASOS; means of '
      'the observations): XMR, Cape Canaveral Skid Strip (2~km from LC-36); TTS, Shuttle Landing Facility (within the '
      'network); COF, Patrick Space Force Base (not used) [24].', 'tab:s_weather', 'lrrrrrr')
write('S3b_speeds', s3b, 'Predicted airwave speed to each working station: still-air speed $c_0$ (Cramer, 1993) for '
      'the mean of XMR and TTS (26.4~$^\\circ$C, 87\\% RH, 1014~hPa), the component of the mean wind along the '
      'source--station ray, the effective speed and the travel time [24].', 'tab:s_speeds', 'lrrrrr')

pk = pd.read_csv(O / '24_origin_time' / 'origin_time_picks.csv')
s4 = pd.DataFrame({'Channel': pk.id.str.replace('1R.', '', regex=False), 'Distance (km)': (pk.distance_m / 1000).map(lambda x: f(x, 2)),
                   'Peak (Pa)': pk.peak_pa.map(lambda x: f(x, 0)), 'Noise (Pa)': pk.noise_pa.map(lambda x: f(x, 2)),
                   'Onset 5\\% (s)': pk.onset_05.map(lambda x: f(x, 2)), 'Onset 10\\% (s)': pk.onset_10.map(lambda x: f(x, 2)),
                   'Onset 20\\% (s)': pk.onset_20.map(lambda x: f(x, 2))})
fit = pd.read_csv(O / '24_origin_time' / 'origin_time_fit.csv')
s4b = pd.DataFrame({'Onset': fit.onset.str.replace('onset_', '', regex=False).str.lstrip('0') + '\\%', 'Model': fit.model,
                    'Origin (UTC)': pd.to_datetime(fit.origin_time).dt.round('10ms').dt.strftime('%H:%M:%S.%f').str[:11], 'Speed (m~s$^{-1}$)': fit.c_mps.map(f),
                    'Spread (s)': fit.spread_s.map(lambda x: f(x, 2)), 'Trend (s~km$^{-1}$)': fit.trend_s_per_km.map(lambda x: f(x, 3))})
write('S4_onsets', s4, 'Airwave onsets at the working infraBSUs, in seconds after the adopted origin '
      '(01:00:02.37~UTC): first sample above 5, 10 or 20\\% of the peak, and never below five times the noise [24].',
      'tab:s_onsets', 'lrrrrrr')
write('S4b_origin_fits', s4b, 'Origin time for each onset definition and speed model: weather-based speed with the '
      'mean wind (adopted with the 5\\% onsets), XMR weather only, no wind, and a straight-line fit with the speed '
      'free. Spread: standard deviation of the station origins; trend: their drift with distance [24].',
      'tab:s_origin', 'llrrrr')

S3_weather (3, 7)
S3b_speeds (4, 6)
S4_onsets (6, 7)
S4b_origin_fits (12, 6)


## S5 Blast parameters and yield, S6 coupling

In [4]:
bp = pd.read_csv(O / '34_amplitude_vs_distance' / 'blast_parameters.csv')
yl = pd.read_csv(O / '34_amplitude_vs_distance' / 'yield_estimates.csv')
s5 = pd.DataFrame({'Channel': bp.id.str.replace('1R.', '', regex=False), '$r$ (km)': bp.distance_km.map(f),
                   '$P_+$ (Pa)': bp.P_pos_Pa.map(lambda x: f(x, 0)), '$P_-$ (Pa)': bp.P_neg_Pa.map(lambda x: f(x, 0)),
                   '$t_+$ (s)': bp.t_pos_s.map(lambda x: f(x, 3)), '$I_+$ (Pa~s)': bp.I_pos_Pa_s.map(f),
                   '0.05--4~Hz (Pa)': bp.P_low_0p05_4Hz_Pa.map(lambda x: f(x, 0)), '4--40~Hz (Pa)': bp.P_high_4_40Hz_Pa.map(lambda x: f(x, 0)),
                   '$Z$ (m~kg$^{-1/3}$)': yl.Z_from_P.map(lambda x: f(x, 0)),
                   '$W(P_+)$ (t)': yl.W_from_P_surface_t.map(lambda x: f(x, 0)), '$W(t_+)$ (t)': yl.W_from_tpos_surface_t.map(lambda x: f(x, 0))})
write('S5_blast', s5, 'Blast-wave parameters ($>$0.05~Hz) and band-limited peaks for each working infraBSU, and the '
      'TNT-equivalent yield $W$ (surface burst) from the Kinney--Graham relations for peak overpressure and '
      'positive-phase duration; $Z$ is the scaled distance implied by $P_+$ [34]. B20 records at 100~samples~s$^{-1}$, '
      'which may lower its $P_+$ by up to $\\sim$28\\%.', 'tab:s_blast', 'lrrrrrrrrrr', size='scriptsize')
dec = pd.read_csv(O / '34_amplitude_vs_distance' / 'decay_fits.csv')
s5b = pd.DataFrame({'Metric': dec.metric.str.replace('|', '$|$', regex=False), 'Data': dec.data,
                    '$n$': dec.n.map(lambda x: f(x, 2)), '$\\sigma_n$': dec.n_se.map(lambda x: f(x, 2)),
                    'Value at 10~km': dec.value_at_10km.map(lambda x: f(x, 2))})
write('S5b_decay', s5b, 'Decay with distance, $A \\propto r^{-n}$, fitted to one value per station (B14: mean of its '
      'three sensors). ``B20 peak corrected\\textquotedblright{} divides B20\\textquoteright s peak by the largest decimation loss (0.72) [34].',
      'tab:s_decay', 'llrrr')

cm = pd.read_csv(O / '28_air_ground_coupling' / 'coupling_metrics.csv')
lr = pd.read_csv(O / '28_air_ground_coupling' / 'layer_resonance.csv').set_index('pair')
s6 = pd.DataFrame({'Pair': cm.pair, 'Band (Hz)': cm.band, '$k_\\mathrm{rms}$': cm.k_rms_um_s_Pa.map(lambda x: f(x, 2)),
                   '$k_\\mathrm{peak}$': cm.k_peak_um_s_Pa.map(lambda x: f(x, 2)), '$C$ (nm~Pa$^{-1}$)': cm.compliance_nm_Pa.map(f),
                   'Lag (s)': cm.compliance_lag_s.map(lambda x: f(x, 2)), '$r$': cm.compliance_r.map(lambda x: f(x, 2)),
                   '$C_0$ (nm~Pa$^{-1}$)': cm.pair.map(lr.C0_nm_Pa).map(lambda x: f(x, 0)),
                   '$f_0$ (Hz)': cm.pair.map(lr.f0_hz).map(f)})
write('S6_coupling', s6, 'Air-to-ground coupling for each co-located infraBSU--Trillium pair: rms and peak ratios of '
      'vertical velocity to pressure (\\si{\\micro\\metre\\per\\second\\per\\pascal}), compliance $C$ from regressing '
      'vertical displacement on pressure (0.5--40~Hz; B20 0.5--12~Hz) with its lag and correlation, low-frequency '
      'spectral compliance $C_0$ (0.5--2.5~Hz, coherent bins) and the ringing frequency $f_0$ after the shock [28]. '
      'B20\\textquoteright s positive $C$ suggests a reversed or faulty vertical.', 'tab:s_coupling', 'llrrrrrrr', size='scriptsize')
a2 = pd.read_csv(O / '28_air_ground_coupling' / 'artemis2_comparison.csv')
s6b = pd.DataFrame({'Source': a2.source, 'Station': a2.station, '$k_\\mathrm{rms}$': a2.k_rms_um_s_Pa.map(lambda x: f(x, 2)),
                    '$k_\\mathrm{peak}$': a2.k_peak_um_s_Pa.map(lambda x: f(x, 2)), 'Peak $p$ (Pa)': a2.peak_p_Pa.map(lambda x: f(x, 0)),
                    '$C$ (nm~Pa$^{-1}$)': a2.compliance_nm_Pa.map(f), '$r$': a2.compliance_r.map(lambda x: f(x, 2))})
write('S6b_artemis', s6b, 'The same coupling measures for the Artemis~II launch (0--180~s after launch) and the New~Glenn '
      'explosion at the two sites that recorded both [28].', 'tab:s_artemis', 'llrrrrr')

S5_blast (6, 11)
S5b_decay (8, 5)
S6_coupling (6, 9)
S6b_artemis (4, 7)


## S7 Seismic arrivals, S8 Chaparral sensitivity, S9 velocity models

In [5]:
sk = pd.read_csv(O / '26_seismic_arrivals' / 'seismic_picks.csv')
s7 = pd.DataFrame({'Station': sk.station, '$r$ (km)': (sk.distance_m / 1000).map(f), 'Airwave (s)': sk.airwave_s.map(f),
                   'AIC 2--12~Hz (s)': sk.aic_2_12_s.map(lambda x: f(x, 2)), 'AIC 1--5~Hz (s)': sk.aic_1_5_s.map(lambda x: f(x, 2)),
                   'First arrival (s)': sk.first_arrival_s.map(lambda x: f(x, 2)), '$v$ (km~s$^{-1}$)': sk.v_first_range_km_s.fillna('--'),
                   'Wavetrain peak (s)': sk.wavetrain_peak_s.map(f), 'Seismic peak (\\si{\\micro\\metre\\per\\second})': sk.seismic_peak_z_um_s.map(f),
                   'Noise (\\si{\\micro\\metre\\per\\second})': sk.noise_rms_z_um_s.map(lambda x: f(x, 2))})
write('S7_seismic', s7, 'Seismic arrivals ahead of the airwave (times after the origin): AIC picks of the first arrival on '
      'the vector amplitude in two bands, their mean and the implied apparent velocity range, the peak of the 2~s-smoothed '
      'wavetrain envelope, and the peak and pre-event rms vertical velocity (2--12~Hz) [26]. At B20 the AIC picks find a '
      'burst within the wavetrain, not a first arrival.', 'tab:s_seismic', 'lrrrrrrrrr', size='scriptsize')

ce = pd.read_csv(O / '32_chaparral_sensitivity' / 'chaparral_epochs.csv')
s8 = pd.DataFrame({'Period (2026)': ce.epoch, 'Windows': ce.n_windows, 'Sources': ce.n_sources,
                   'DDF/DD1 median': ce.ratio_median.map(lambda x: f(x, 0)),
                   'IQR': ce.ratio_p25.map(lambda x: f(x, 0)) + '--' + ce.ratio_p75.map(lambda x: f(x, 0)),
                   '$S_\\mathrm{M25}$ (V~Pa$^{-1}$)': ce.S_median_V_Pa.map(lambda x: f(x, 3))})
write('S8_chaparral', s8, 'Sensitivity of the B20 Chaparral M25 relative to the co-located infraBSU (46~\\si{\\micro\\volt'
      '\\per\\pascal}) from the ratio of their raw counts (0.3--12~Hz) in coherent, unclipped 30~s windows after the 2026 '
      'launches and around the explosion, by period between field visits [32].', 'tab:s_chaparral', 'lrrrrr')

rows = []
for nbk, fname, label in (('36_velocity_model', 'best_model_B.csv', 'Verticals only (+ resonance)'),
                          ('38_horizontal_components', 'best_model_C.csv', 'C: Rayleigh + Love + P'),
                          ('38_horizontal_components', 'best_model_D.csv', 'D: C + H/V')):
    m = pd.read_csv(O / nbk / fname, index_col=0)
    z = 0.0
    for name, r in m.iterrows():
        rows.append({'Model': label, 'Layer': name, 'Top (m)': f(z, 0),
                     'Thickness (m)': f(r.thickness_m, 0) if r.thickness_m > 0 else '$\\infty$',
                     '$V_P$ (m~s$^{-1}$)': f(r.vp_m_s, 0), '$V_S$ (m~s$^{-1}$)': f(r.vs_m_s, 0), '$\\rho$ (kg~m$^{-3}$)': f(r.rho_kg_m3, 0)})
        z += r.thickness_m
s9 = pd.DataFrame(rows)
write('S9_models', s9, 'Best-fitting flat velocity models: from the vertical-component Rayleigh dispersion, P first arrivals '
      'and the ground resonance [36]; and from Rayleigh and Love dispersion and P first arrivals without (C) and with (D) '
      'the Rayleigh H/V ratio [38]. $V_P$ and $\\rho$ are fixed by assumption (see text).', 'tab:s_models', 'llrrrrr')

jd = pd.read_csv(O / '38_horizontal_components' / 'joint_data.csv')
piv = {}
for kind in ('Rayleigh', 'Love', 'H/V'):
    g = jd[jd.kind == kind].set_index('f_hz')
    scale, dgt = (1000.0, 2) if kind != 'H/V' else (1.0, 2)
    piv[kind] = (g.value / scale).map(lambda x: f(x, dgt)) + ' $\\pm$ ' + (g.sigma / scale).map(lambda x: f(x, dgt))
s10 = pd.DataFrame(piv).reset_index().rename(columns={'f_hz': 'Frequency (Hz)', 'Rayleigh': '$U_R$ (km~s$^{-1}$)',
                                                       'Love': '$U_L$ (km~s$^{-1}$)', 'H/V': 'H/V'})
s10['Frequency (Hz)'] = s10['Frequency (Hz)'].map(lambda x: f(x, 2))
write('S10_dispersion', s10, 'Dispersion data for the joint model: Rayleigh group velocity (median of Z at B14, B29, B05 and '
      'R at B14, B29), Love group velocity (T at B14 and B29; only $\\leq$1.8~Hz is fitted) and the Rayleigh H/V ridge '
      'amplitude ratio (B14 and B29), with their uncertainties [38].', 'tab:s_dispersion', 'rlll', size='scriptsize')

S7_seismic (4, 10)
S8_chaparral (3, 6)
S9_models (11, 7)
S10_dispersion (22, 4)
